# 🔍 Датасеты для обучения детекции контекстных галлюцинаций

В этом ноутбуке загружаем и исследуем три топовых датасета с HuggingFace для задачи обнаружения **контекстных галлюцинаций** — ситуаций, когда модель генерирует ответ, не поддержанный предоставленным контекстом.

| # | Датасет | Тип | Размер | Формат |
|---|---------|-----|--------|--------|
| 1 | **RAGTruth** (`wandb/RAGTruth-processed`) | RAG QA/Summarization | ~18K | context + question + answer + hallucination spans |
| 2 | **HalluMix** (`quotientai/HalluMix`) | Multi-domain | 6.5K | context (chunks) + answer + binary label |
| 3 | **SQuAD v2** (`rajpurkar/squad_v2`) | Reading Comprehension | 150K | context + question + answer (+ unanswerable) |

## 📦 Установка зависимостей

In [ ]:
!pip install datasets pandas matplotlib seaborn -q

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from datasets import load_dataset
import json
import textwrap
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 200)
pd.set_option('display.max_columns', 20)

# Цветовая схема для визуализаций
COLORS = {
    'faithful': '#2ecc71',
    'hallucinated': '#e74c3c',
    'neutral': '#3498db',
    'accent': '#9b59b6'
}

print('✅ Импорты успешны')

---
## 📚 Датасет 1: RAGTruth

**Описание:** Корпус ~18,000 примеров для анализа **слово-уровневых галлюцинаций** в RAG-системах. Ответы сгенерированы разными LLM (GPT-4, GPT-3.5, LLaMA-2, Mistral) и аннотированы вручную.

**Задачи:**
- QA (на основе MS MARCO)
- Data-to-Text (Yelp reviews)
- Summarization

**Ключевые поля:** `source_info` (контекст), `question`, `response` (ответ модели), `labels` (аннотации галлюцинаций)

📌 *Лучший датасет для тонкой настройки детектора галлюцинаций на уровне токенов/спанов*

In [ ]:
print('⏳ Загружаем RAGTruth...')
ragtruth_ds = load_dataset('wandb/RAGTruth-processed', trust_remote_code=True)
print('✅ RAGTruth загружен!')
print()
print('📊 Сплиты:', ragtruth_ds)


In [ ]:
# Структура датасета
split_name = list(ragtruth_ds.keys())[0]
rt_df = ragtruth_ds[split_name].to_pandas()

print(f'🔢 Размер ({split_name}): {len(rt_df):,} строк')
print(f'📋 Колонки: {list(rt_df.columns)}')
print()
rt_df.head(3)

In [ ]:
# Детальный просмотр типов данных и пропусков
print('📈 Info:')
rt_df.info()
print()
print('📊 Уникальные значения по колонкам:')
for col in rt_df.columns:
    n_unique = rt_df[col].nunique()
    print(f'  {col}: {n_unique} уникальных значений')

In [ ]:
# Смотрим один конкретный пример
print('=' * 80)
print('📝 ПРИМЕР ИЗ RAGTruth')
print('=' * 80)

sample = rt_df.iloc[0]
for col in rt_df.columns:
    val = sample[col]
    print(f'\n🔹 {col.upper()}:')
    if isinstance(val, str) and len(val) > 300:
        print(textwrap.fill(val[:300] + '...', width=90))
    elif isinstance(val, (dict, list)):
        print(json.dumps(val, indent=2, ensure_ascii=False)[:500])
    else:
        print(val)

In [ ]:
# Визуализация RAGTruth
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('RAGTruth — Анализ датасета', fontsize=14, fontweight='bold')

# Распределение меток (если есть)
label_col = None
for candidate in ['has_hallucination', 'label', 'hallucinated', 'is_hallucination']:
    if candidate in rt_df.columns:
        label_col = candidate
        break

if label_col:
    counts = rt_df[label_col].value_counts()
    axes[0].bar(counts.index.astype(str), counts.values, 
                color=[COLORS['faithful'], COLORS['hallucinated']][:len(counts)])
    axes[0].set_title(f'Распределение меток ({label_col})')
    axes[0].set_xlabel('Метка')
    axes[0].set_ylabel('Количество')
    for i, v in enumerate(counts.values):
        axes[0].text(i, v + 50, f'{v:,}\n({100*v/len(rt_df):.1f}%)', ha='center', fontsize=10)
else:
    axes[0].text(0.5, 0.5, 'Нет явного столбца с меткой', ha='center', va='center', fontsize=12)
    axes[0].set_title('Метки')

# Длина текстов
text_col = None
for candidate in ['response', 'answer', 'output', 'text', 'generated_text']:
    if candidate in rt_df.columns:
        text_col = candidate
        break

if text_col:
    lengths = rt_df[text_col].dropna().str.len()
    axes[1].hist(lengths.clip(upper=lengths.quantile(0.95)), bins=40, 
                 color=COLORS['neutral'], alpha=0.8, edgecolor='white')
    axes[1].set_title(f'Длина поля "{text_col}" (символы)')
    axes[1].set_xlabel('Длина')
    axes[1].set_ylabel('Частота')
    axes[1].axvline(lengths.median(), color=COLORS['hallucinated'], 
                   linestyle='--', label=f'Медиана: {lengths.median():.0f}')
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, 'Текстовая колонка не найдена', ha='center', va='center')

plt.tight_layout()
plt.show()
print(f'\n✅ RAGTruth: всего {len(rt_df):,} примеров, {len(rt_df.columns)} колонок')

---
## 📚 Датасет 2: HalluMix

**Описание:** Task-agnostic многодоменный бенчмарк для оценки детекции галлюцинаций в реалистичных RAG-сценариях.

**Особенности:**
- 6,500 примеров
- Домены: healthcare, law, science, news
- Задачи: QA, summarization, NLI
- Контекст — список перемешанных чанков (как в реальном RAG!), включая нерелевантные

**Ключевые поля:** `documents` (контекст-чанки), `answer` (гипотеза), `hallucination` (бинарная метка)

📌 *Лучший для оценки обобщаемости детектора — несколько доменов и задач*

In [ ]:
print('⏳ Загружаем HalluMix...')
hallumix_ds = load_dataset('quotientai/HalluMix', trust_remote_code=True)
print('✅ HalluMix загружен!')
print()
print('📊 Сплиты:', hallumix_ds)

In [ ]:
hm_split = list(hallumix_ds.keys())[0]
hm_df = hallumix_ds[hm_split].to_pandas()

print(f'🔢 Размер ({hm_split}): {len(hm_df):,} строк')
print(f'📋 Колонки: {list(hm_df.columns)}')
print()
hm_df.head(3)

In [ ]:
# Детальный анализ структуры
print('📈 Типы данных:')
print(hm_df.dtypes)
print()
print('📊 Пропуски:')
print(hm_df.isnull().sum())
print()
# Если есть source_dataset — посмотрим откуда данные
for candidate in ['source', 'source_dataset', 'source_identifier', 'task', 'domain']:
    if candidate in hm_df.columns:
        print(f'\n📌 Распределение по "{candidate}":')
        print(hm_df[candidate].value_counts())
        break

In [ ]:
# Просмотр примера
print('=' * 80)
print('📝 ПРИМЕР ИЗ HalluMix')
print('=' * 80)

sample = hm_df.iloc[0]
for col in hm_df.columns:
    val = sample[col]
    print(f'\n🔹 {col.upper()}:')
    if isinstance(val, list):
        # Если список чанков — выводим первые 2
        for i, chunk in enumerate(val[:2]):
            print(f'  [Чанк {i+1}]: {str(chunk)[:200]}')
        if len(val) > 2:
            print(f'  ... и ещё {len(val)-2} чанков')
    elif isinstance(val, str) and len(val) > 300:
        print(textwrap.fill(val[:300] + '...', width=90))
    else:
        print(val)

In [ ]:
# Визуализация HalluMix
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('HalluMix — Анализ датасета', fontsize=14, fontweight='bold')

# 1. Метки
label_col = None
for c in ['hallucination', 'label', 'hallucinated', 'is_hallucination']:
    if c in hm_df.columns:
        label_col = c
        break

if label_col:
    counts = hm_df[label_col].value_counts()
    colors = [COLORS['faithful'] if str(k) in ['0', 'False', 'faithful'] 
              else COLORS['hallucinated'] for k in counts.index]
    wedges, texts, autotexts = axes[0].pie(
        counts.values, labels=[f'Метка {k}' for k in counts.index],
        autopct='%1.1f%%', colors=colors, startangle=90
    )
    axes[0].set_title(f'Баланс классов ({label_col})')
    green_patch = mpatches.Patch(color=COLORS['faithful'], label='Faithful (0)')
    red_patch = mpatches.Patch(color=COLORS['hallucinated'], label='Hallucinated (1)')
    axes[0].legend(handles=[green_patch, red_patch], loc='lower left', fontsize=8)

# 2. Количество чанков в контексте
doc_col = None
for c in ['documents', 'context', 'passages', 'chunks']:
    if c in hm_df.columns:
        doc_col = c
        break

if doc_col:
    chunk_counts = hm_df[doc_col].apply(lambda x: len(x) if isinstance(x, list) else 1)
    axes[1].hist(chunk_counts, bins=range(1, chunk_counts.max()+2), 
                color=COLORS['accent'], alpha=0.8, edgecolor='white', align='left')
    axes[1].set_title('Количество чанков в контексте')
    axes[1].set_xlabel('Число чанков')
    axes[1].set_ylabel('Частота')

# 3. Длина ответов
answer_col = None
for c in ['answer', 'response', 'hypothesis', 'output', 'text']:
    if c in hm_df.columns and hm_df[c].dtype == object:
        answer_col = c
        break

if answer_col and label_col:
    for label_val, color, name in [(0, COLORS['faithful'], 'Faithful'),
                                   (1, COLORS['hallucinated'], 'Hallucinated')]:
        subset = hm_df[hm_df[label_col] == label_val][answer_col].dropna().str.len()
        if len(subset):
            axes[2].hist(subset.clip(upper=subset.quantile(0.95)), bins=30,
                        alpha=0.6, color=color, label=name, edgecolor='white')
    axes[2].set_title('Длина ответов по классам')
    axes[2].set_xlabel('Длина (символы)')
    axes[2].set_ylabel('Частота')
    axes[2].legend()

plt.tight_layout()
plt.show()
print(f'\n✅ HalluMix: всего {len(hm_df):,} примеров')

---
## 📚 Датасет 3: SQuAD v2

**Описание:** Золотой стандарт для QA и обнаружения галлюцинаций. SQuAD v2 добавляет **неотвечаемые вопросы** — модель должна распознать, когда ответа нет в контексте и не галлюцинировать его.

**Особенности:**
- 150,000+ примеров (train) + 20,000 (validation)
- ~50% вопросов — unanswerable (ответ не поддерживается контекстом)
- Контекст — параграфы из Википедии
- Идеален для **оценки** и **fine-tuning** на задачу «есть ли ответ в контексте»

📌 *Лучший baseline-датасет: огромный, хорошо изученный, прямая связь с галлюцинацией через unanswerable questions*

In [ ]:
print('⏳ Загружаем SQuAD v2...')
squad_ds = load_dataset('rajpurkar/squad_v2')
print('✅ SQuAD v2 загружен!')
print()
print('📊 Сплиты:', squad_ds)

In [ ]:
sq_train = squad_ds['train'].to_pandas()
sq_val = squad_ds['validation'].to_pandas()

print(f'🔢 Train: {len(sq_train):,} строк')
print(f'🔢 Validation: {len(sq_val):,} строк')
print(f'📋 Колонки: {list(sq_train.columns)}')
print()
sq_train.head(3)

In [ ]:
# Добавляем флаг answerable
sq_train['is_answerable'] = sq_train['answers'].apply(
    lambda x: len(x['text']) > 0 if isinstance(x, dict) else False
)
sq_val['is_answerable'] = sq_val['answers'].apply(
    lambda x: len(x['text']) > 0 if isinstance(x, dict) else False
)

print('📊 Train — answerable / unanswerable:')
print(sq_train['is_answerable'].value_counts())
print()
print('📊 Validation — answerable / unanswerable:')
print(sq_val['is_answerable'].value_counts())

In [ ]:
# Пример answerable
print('=' * 80)
print('✅ ПРИМЕР: Answerable question (правильный ответ ЕСТЬ в контексте)')
print('=' * 80)
sample_ans = sq_train[sq_train['is_answerable'] == True].iloc[0]
print(f'\n📖 CONTEXT (первые 400 симв.):')
print(textwrap.fill(sample_ans['context'][:400], width=90))
print(f'\n❓ QUESTION: {sample_ans["question"]}')
print(f'\n✅ ANSWER: {sample_ans["answers"]["text"]}')

print()
print('=' * 80)
print('❌ ПРИМЕР: Unanswerable question (ответ НЕ поддерживается контекстом)')
print('=' * 80)
sample_unans = sq_train[sq_train['is_answerable'] == False].iloc[0]
print(f'\n📖 CONTEXT (первые 400 симв.):')
print(textwrap.fill(sample_unans['context'][:400], width=90))
print(f'\n❓ QUESTION: {sample_unans["question"]}')
print(f'\n🚫 ANSWERS: {sample_unans["answers"]} (пустой список → модель не должна галлюцинировать!)')

In [ ]:
# Визуализация SQuAD v2
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('SQuAD v2 — Анализ датасета', fontsize=14, fontweight='bold')

# 1. Train vs Val баланс классов
for ax, df, title in [(axes[0], sq_train, 'Train'), (axes[1], sq_val, 'Validation')]:
    counts = df['is_answerable'].value_counts()
    labels = ['Answerable\n(нет галлюцинации)', 'Unanswerable\n(потенц. галлюцинация)']
    bars = ax.bar(labels, 
                  [counts.get(True, 0), counts.get(False, 0)],
                  color=[COLORS['faithful'], COLORS['hallucinated']], 
                  alpha=0.85, edgecolor='white', linewidth=1.5)
    ax.set_title(f'{title} split')
    ax.set_ylabel('Количество примеров')
    total = len(df)
    for bar, v in zip(bars, [counts.get(True, 0), counts.get(False, 0)]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
               f'{v:,}\n({100*v/total:.1f}%)', ha='center', fontsize=9)
    ax.set_ylim(0, max(counts.values) * 1.15)

# 2. Длина контекстов
ctx_lens = sq_train['context'].str.len()
axes[2].hist(ctx_lens.clip(upper=ctx_lens.quantile(0.95)), bins=40,
            color=COLORS['neutral'], alpha=0.8, edgecolor='white')
axes[2].set_title('Длина контекстов (Train)')
axes[2].set_xlabel('Длина (символы)')
axes[2].set_ylabel('Частота')
axes[2].axvline(ctx_lens.median(), color='red', linestyle='--',
               label=f'Медиана: {ctx_lens.median():.0f}')
axes[2].legend()

plt.tight_layout()
plt.show()
print(f'\n✅ SQuAD v2: {len(sq_train):,} train / {len(sq_val):,} val примеров')

---
## 📊 Сравнительный анализ датасетов

In [ ]:
# Итоговая таблица сравнения
summary = pd.DataFrame([
    {
        'Датасет': 'RAGTruth',
        'HuggingFace ID': 'wandb/RAGTruth-processed',
        'Размер': f'~{len(rt_df):,}',
        'Тип задачи': 'QA / Summ / Data2Text',
        'Тип галлюцинации': 'Span-level (слово/фраза)',
        'Источник контекста': 'MS MARCO, Yelp, CNN',
        'Аннотация': 'Ручная (span-level)',
        'Рекомендуется для': 'Fine-tuning детектора'
    },
    {
        'Датасет': 'HalluMix',
        'HuggingFace ID': 'quotientai/HalluMix',
        'Размер': f'~{len(hm_df):,}',
        'Тип задачи': 'QA / NLI / Summ (mixed)',
        'Тип галлюцинации': 'Binary (пример целиком)',
        'Источник контекста': 'Multi-domain (law, health, sci)',
        'Аннотация': 'Автоматическая (из NLI/QA)',
        'Рекомендуется для': 'Evaluation / benchmarking'
    },
    {
        'Датасет': 'SQuAD v2',
        'HuggingFace ID': 'rajpurkar/squad_v2',
        'Размер': f'~{len(sq_train)+len(sq_val):,}',
        'Тип задачи': 'Reading Comprehension QA',
        'Тип галлюцинации': 'Binary unanswerable',
        'Источник контекста': 'Wikipedia',
        'Аннотация': 'Ручная (краудсорс)',
        'Рекомендуется для': 'Baseline / pre-training'
    }
])

summary.set_index('Датасет', inplace=True)
print('📋 СРАВНИТЕЛЬНАЯ ТАБЛИЦА ДАТАСЕТОВ')
print('=' * 80)
summary.T

In [ ]:
# Финальная визуализация: размеры датасетов
fig, ax = plt.subplots(figsize=(10, 5))

datasets = ['RAGTruth', 'HalluMix', 'SQuAD v2']
sizes = [len(rt_df), len(hm_df), len(sq_train) + len(sq_val)]
colors = [COLORS['hallucinated'], COLORS['accent'], COLORS['neutral']]

bars = ax.barh(datasets, sizes, color=colors, alpha=0.85, edgecolor='white', linewidth=2)

for bar, size in zip(bars, sizes):
    ax.text(bar.get_width() + max(sizes)*0.01, bar.get_y() + bar.get_height()/2,
           f'{size:,} примеров', va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Количество примеров', fontsize=12)
ax.set_title('Размеры датасетов для детекции контекстных галлюцинаций', 
            fontsize=13, fontweight='bold', pad=15)
ax.set_xlim(0, max(sizes) * 1.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
print('\n🎯 Итог: все три датасета загружены и исследованы!')

---
## 🎯 Рекомендации по использованию

### Стратегия обучения:

1. **Pre-train/baseline** → SQuAD v2
   - Огромный размер, качественная аннотация
   - Учит модель задавать вопрос: «поддерживается ли этот ответ контекстом?»

2. **Fine-tune** → RAGTruth
   - Реальные RAG-ответы от LLM
   - Span-level аннотация позволяет обучить точный детектор
   - Несколько задач (QA + Summarization + Data2Text)

3. **Evaluation / out-of-domain test** → HalluMix
   - Многодоменный: проверяет обобщаемость
   - Реалистичный шум (нерелевантные чанки) — как в боевом RAG

### Метрики для оценки:
- **Binary detection**: F1, ROC-AUC
- **Span-level**: Token-F1, span-overlap
- **Calibration**: ECE (Expected Calibration Error)